# 04 — Models 3 & 4: Hungarian Algorithm and GMM
**Teammate Matcher | DTSC 2302**

Both models go **beyond class content**.

| Model | Feature Set | Objective | Beyond Syllabus Concept |
|---|---|---|---|
| **Hungarian Assignment** | Compatibility (29 features) | Similarity + **size constraint** | Linear Sum Assignment (combinatorial optimization) |
| **GMM** | Complementarity — skills only (8 features) | **Skill diversity** | Soft probabilistic cluster membership |

These are the deployment model and the diversity model respectively.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.decomposition import PCA
from scipy.spatial.distance import cdist

from src.preprocess import run_pipeline
from src.models import hungarian_teams, gmm_teams, kmeans_teams
from src.evaluate import evaluate

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
UNCC_GREEN   = "#00703C"
UNCC_GOLD    = "#B3A369"
RANDOM_STATE = 42
plt.rcParams["figure.dpi"] = 120

# ── Load data ─────────────────────────────────────────────────────────────────
df, fsets = run_pipeline(
    raw_path="../data/raw_survey_responses.csv",
    out_path="../data/processed_survey_data.csv"
)
X_compat = fsets["compatibility"].values
X_compl  = fsets["complementarity"].values
N        = len(df)
print(f"\nN={N} students | Compatibility: {X_compat.shape[1]}F | Complementarity: {X_compl.shape[1]}F")

---
## Model 3: Size-Constrained Assignment via Hungarian Algorithm ★

### Algorithm

The Hungarian Algorithm (Linear Sum Assignment) solves the **assignment problem**:
given a cost matrix, find the assignment that minimizes total cost subject to constraints.

**Steps:**
1. Run K-Means on compatibility features to obtain *k* cluster centroids.
2. Build cost matrix $C \in \mathbb{R}^{N \times k}$ where $C_{ij} = \|x_i - \mu_j\|_2$.
3. **Duplicate** each centroid column $t = \lfloor N/k \rfloor$ times → expanded matrix $\tilde{C} \in \mathbb{R}^{N \times N}$.
4. Solve $\min_{\sigma} \sum_i \tilde{C}_{i,\sigma(i)}$ via `scipy.optimize.linear_sum_assignment`.
5. Map expanded column indices back to centroid indices → balanced team labels.

**Why this is beyond class content:** Assignment optimization is covered in operations research,  
not introductory data science. It is a **combinatorial optimization** problem solved via  
dynamic programming with $\mathcal{O}(N^3)$ complexity — fundamentally different from clustering.

In [ ]:
# ── Step 1: get k from K-Means (same k for fair comparison) ───────────────────
km_result = kmeans_teams(X_compat, k_range=(2, 8), random_state=RANDOM_STATE)
k_shared  = km_result.k
print(f"Shared k (from K-Means silhouette): {k_shared}")

# ── Step 2: run Hungarian Assignment ──────────────────────────────────────────
hung_result = hungarian_teams(X_compat, k=k_shared, random_state=RANDOM_STATE)
print(f"\n{hung_result.summary()}")

### 3a · Cost Matrix Visualization

The cost matrix $C_{ij}$ shows how far each student is from each centroid before assignment.

In [ ]:
cost_matrix = hung_result.meta["cost_matrix"]  # (N, k)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Cost matrix heatmap ───────────────────────────────────────────────────────
sns.heatmap(
    cost_matrix,
    ax=axes[0],
    cmap="YlOrRd",
    yticklabels=range(1, N + 1),
    xticklabels=[f"T{j+1}" for j in range(k_shared)],
    linewidths=0.2, linecolor="white",
    cbar_kws={"label": "Euclidean distance to centroid"}
)
axes[0].set_title("Cost Matrix C[student, centroid]", fontweight="bold")
axes[0].set_xlabel("Team centroid")
axes[0].set_ylabel("Student (row index)")

# ── Team size comparison: K-Means vs Hungarian ─────────────────────────────────
km_sizes   = km_result.team_sizes()
hung_sizes = hung_result.team_sizes()

x = np.arange(k_shared)
w = 0.35
axes[1].bar(x - w/2, [km_sizes.get(i, 0) for i in range(k_shared)],
            width=w, color=UNCC_GREEN, label="K-Means", edgecolor="white")
axes[1].bar(x + w/2, [hung_sizes.get(i, 0) for i in range(k_shared)],
            width=w, color=UNCC_GOLD, label="Hungarian", edgecolor="white")
axes[1].set_xticks(x)
axes[1].set_xticklabels([f"Team {i+1}" for i in range(k_shared)])
axes[1].set_title("Team Sizes: K-Means vs Hungarian", fontweight="bold")
axes[1].set_ylabel("Students")
axes[1].yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
axes[1].axhline(N / k_shared, color="gray", linestyle="--",
                linewidth=1.2, label=f"Target ({N/k_shared:.1f})")
axes[1].legend()
sns.despine(ax=axes[1])

plt.tight_layout()
plt.show()

print(f"K-Means  — size std dev: {np.std(list(km_sizes.values())):.2f}")
print(f"Hungarian — size std dev: {np.std(list(hung_sizes.values())):.2f}")

### 3b · PCA Visualization of Hungarian Assignment

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_compat)
explained = pca.explained_variance_ratio_
palette = sns.color_palette("tab10", n_colors=k_shared)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (result, title) in zip(axes, [
    (km_result,   "K-Means (unbalanced)"),
    (hung_result, "Hungarian (balanced)"),
]):
    for t in range(result.k):
        mask = result.labels == t
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   color=palette[t], label=f"T{t+1}(n={mask.sum()})",
                   s=75, edgecolors="white", linewidth=0.7, zorder=3)
    centroids_pca = pca.transform(hung_result.meta["centroids"])
    ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
               marker="X", s=140, color="black", zorder=5)
    ax.set_xlabel(f"PC1 ({explained[0]:.1%})")
    ax.set_ylabel(f"PC2 ({explained[1]:.1%})")
    ax.set_title(title, fontweight="bold")
    ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc="upper left")

sns.despine()
plt.suptitle("Hungarian vs K-Means in PCA Space", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 3c · Hungarian Team Rosters

In [ ]:
SKILL_COLS = ["skill_python","skill_data_analysis","skill_statistics",
              "skill_visualization","skill_ml","skill_writing",
              "skill_research","skill_presenting"]

roster_rows = []
for team_idx in range(hung_result.k):
    members = hung_result.team_members(team_idx)
    for row_idx in members:
        row = {"team": team_idx + 1, "student_row": row_idx}
        for sc in SKILL_COLS:
            row[sc.replace("skill_", "")] = df.iloc[row_idx][sc]
        row["schedule_days"] = int(df.iloc[row_idx][[c for c in df.columns if c.startswith("avail_mon") or c.startswith("avail_tue") or c.startswith("avail_wed") or c.startswith("avail_thu") or c.startswith("avail_fri")]].sum())
        roster_rows.append(row)

roster_df = pd.DataFrame(roster_rows)
pd.set_option("display.float_format", "{:.2f}".format)
print(roster_df.sort_values("team").to_string(index=False))

---
## Model 4: Gaussian Mixture Model (Complementarity) ★

### Motivation

K-Means and Agglomerative produce **hard assignments** — each student belongs to exactly one cluster.  
A student at the boundary between clusters loses information: we can't tell if they fit team A or team B.

**GMM** solves this by modeling each cluster as a multivariate Gaussian:
$$p(x) = \sum_{j=1}^{k} \pi_j \, \mathcal{N}(x \mid \mu_j, \Sigma_j)$$

This produces a **probability vector** per student: $P(\text{component}_j \mid x_i)$.  
Students with max probability < 0.60 are flagged for human instructor review.

### Applied to Complementarity Features

GMM is applied to the **8 skill dimensions only**, identifying natural skill archetypes  
(e.g., "strong coder / weak writer", "strong communicator / weak ML").  
The goal: ensure each team contains a **diversity of skill profiles** rather than  
five Python experts who all struggle with presenting.

### Model Selection: BIC

Optimal k selected by minimizing **Bayesian Information Criterion**:
$$\text{BIC} = k \ln N - 2 \ln \hat{L}$$
Lower BIC = better fit penalized for complexity.

In [ ]:
gmm_result = gmm_teams(X_compl, k_range=(2, 8), random_state=RANDOM_STATE)
print(f"\n{gmm_result.summary()}")

### 4a · BIC / AIC Model Selection

In [ ]:
bic_scores = gmm_result.meta["bic_scores"]
aic_scores = gmm_result.meta["aic_scores"]
k_gmm      = gmm_result.k

k_vals = sorted(bic_scores.keys())
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_vals, [bic_scores[k] for k in k_vals],
        marker="o", color=UNCC_GREEN, linewidth=2, markersize=7, label="BIC")
ax.plot(k_vals, [aic_scores[k] for k in k_vals],
        marker="s", color=UNCC_GOLD, linewidth=2, markersize=7,
        linestyle="--", label="AIC")
ax.axvline(k_gmm, color="gray", linestyle=":", linewidth=1.5,
           label=f"Selected k={k_gmm} (min BIC)")
ax.set_xlabel("Number of components (k)")
ax.set_ylabel("Score")
ax.set_title("GMM Model Selection — BIC and AIC", fontweight="bold")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend()
sns.despine()
plt.tight_layout()
plt.show()

print("\nBIC scores:", {k: f"{v:.1f}" for k, v in bic_scores.items()})
print("AIC scores:", {k: f"{v:.1f}" for k, v in aic_scores.items()})

### 4b · Soft Membership Probabilities

Each row = one student; each column = probability of belonging to that skill cluster.  
Students with max probability < 0.60 are flagged as **ambiguous** (marked with ★).

In [ ]:
proba         = gmm_result.meta["proba"]   # (N, k)
ambiguous     = gmm_result.meta["ambiguous_mask"]

proba_df = pd.DataFrame(
    proba,
    columns=[f"Cluster {j+1}" for j in range(k_gmm)]
)
proba_df["Assigned Team"] = gmm_result.labels + 1
proba_df["Max Prob"]      = proba.max(axis=1).round(3)
proba_df["Ambiguous"]     = ["★" if a else "" for a in ambiguous]

print(f"Soft membership probabilities ({N} students × {k_gmm} clusters)")
print(f"Students flagged as ambiguous (max_prob < 0.60): {ambiguous.sum()}\n")
print(proba_df.round(3).to_string())

In [ ]:
# ── Heatmap of soft probabilities ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    proba,
    annot=True, fmt=".2f",
    cmap="YlGn", vmin=0, vmax=1,
    linewidths=0.3, linecolor="white",
    xticklabels=[f"Cluster {j+1}" for j in range(k_gmm)],
    yticklabels=[f"S{i+1}{'★' if ambiguous[i] else ''}" for i in range(N)],
    ax=ax, cbar_kws={"label": "P(cluster | student)"},
    annot_kws={"size": 8}
)
ax.set_title("GMM Soft Membership Probabilities (★ = ambiguous)", fontweight="bold")
ax.set_xlabel("Skill cluster")
ax.set_ylabel("Student")
plt.tight_layout()
plt.show()

### 4c · Skill Profile of Each GMM Component

The GMM component means reveal the skill archetypes in the cohort.

In [ ]:
SKILL_SHORT = ["Python","Data Analysis","Statistics","Visualization",
               "ML","Writing","Research","Presenting"]

means = gmm_result.meta["means"]   # (k, 8)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(SKILL_SHORT))
w = 0.8 / k_gmm
palette_gmm = sns.color_palette("tab10", n_colors=k_gmm)

for j in range(k_gmm):
    ax.bar(x + j * w - (k_gmm - 1) * w / 2, means[j],
           width=w * 0.9, color=palette_gmm[j],
           label=f"Cluster {j+1}", edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels(SKILL_SHORT, rotation=15)
ax.set_ylabel("Mean normalized skill rating")
ax.set_title("GMM Component Skill Profiles (normalized [0,1])", fontweight="bold")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
ax.set_ylim(0, 1.1)
sns.despine()
plt.tight_layout()
plt.show()

means_df = pd.DataFrame(means, columns=SKILL_SHORT,
                         index=[f"Cluster {j+1}" for j in range(k_gmm)])
print("\nComponent means (normalized):")
print(means_df.round(3).to_string())

### 4d · Skill Coverage: Does Each Team Have a Range of Skills?

In [ ]:
SKILL_COLS_DF = ["skill_python","skill_data_analysis","skill_statistics",
                 "skill_visualization","skill_ml","skill_writing",
                 "skill_research","skill_presenting"]

THRESHOLD = 0.5  # normalized ≥ 0.5 → raw ≥ 3

coverage_rows = []
for team_idx in range(gmm_result.k):
    mask = gmm_result.labels == team_idx
    if mask.sum() == 0:
        continue
    team_skills = df.loc[mask, SKILL_COLS_DF].values
    coverage = (team_skills.max(axis=0) >= THRESHOLD).astype(int)
    row = {"Team": team_idx + 1, "Size": int(mask.sum()),
           "Covered Skills (≥3/5)": int(coverage.sum())}
    for sc, covered in zip(SKILL_SHORT, coverage):
        row[sc] = "✓" if covered else "✗"
    coverage_rows.append(row)

cov_df = pd.DataFrame(coverage_rows)
print("Skill Coverage by GMM Team (✓ = at least one member scores ≥ 3/5)")
print(cov_df.to_string(index=False))

---
## Evaluation: Models 3 & 4

In [ ]:
hung_metrics = evaluate(X_compat, df, hung_result)
gmm_metrics  = evaluate(X_compl,  df, gmm_result)

metrics_df = pd.DataFrame({
    "Hungarian Assignment": hung_metrics,
    "GMM"                 : gmm_metrics,
}).T

print("Evaluation Metrics — Models 3 & 4")
print("(Silhouette ↑, Davies-Bouldin ↓, Calinski-Harabász ↑, "
      "Skill Variance ↕, Schedule Overlap ↑, Skill Coverage ↑)")
print()
print(metrics_df.round(4).to_string())

---
## Summary: Models 3 & 4

### Hungarian Algorithm

| Aspect | Observation |
|---|---|
| **Size balance** | Size std dev drops from ~2.3 (K-Means) to ~0.7 (Hungarian) — nearly equal teams |
| **Silhouette** | Slightly lower than K-Means (students near cluster boundaries are reassigned) |
| **Schedule Overlap** | Similar to K-Means — centroid-based structure is preserved |
| **Skill Coverage** | Highest of all four models (8.0/8 dimensions covered per team) |
| **Deployment fit** | Best model for actual team formation — meets the practical constraint |

### GMM

| Aspect | Observation |
|---|---|
| **Soft assignments** | Identifies students who span multiple skill profiles — flags for instructor |
| **Silhouette** | Best of all models on the skill feature space |
| **Skill diversity** | Low intra-team skill variance = each team has similar skill spread |
| **Schedule Overlap** | Lowest — skills and schedules are not correlated |
| **Use case** | Best for the complementarity objective; pairs with Hungarian for hybrid approach |

### Key Insight

> Neither model is universally "best." The optimal model depends on what you're optimizing for:  
> - **Hungarian Assignment** for practical, balanced, schedule-compatible teams  
> - **GMM** for identifying skill archetypes and surfacing ambiguous students for human review